In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf latex2sympy2 sympy faiss-cpu trafilatura

In [ ]:
import os, sys, re, time, torch, contextlib, io, textwrap
import sympy as sp
from sympy import symbols, simplify, N

BASE_DIR    = '/content/gdrive/MyDrive/NLP_assignment'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

if not os.path.exists(BASE_DIR):
    print(f"Error path {BASE_DIR} not found. Please check your Google Drive paths.")

sys.path.append(BASE_DIR)
print("Environment ready")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, gc

PLANNER_ID = "Qwen/Qwen2.5-7B-Instruct"

print("Loading 7B planner Qwen in four bit mode")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(PLANNER_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    PLANNER_ID,
    quantization_config=bnb_config,
    device_map="auto"
).eval()

print(f"Planner model loaded {PLANNER_ID}")
if torch.cuda.is_available():
    print(f"GPU memory allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
import concurrent.futures
import json
import builtins
import os
import re
import textwrap
import urllib.parse
import warnings
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta

import faiss
import requests
import trafilatura
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer


# Keep notebook output clean while the helper libraries load and run.
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.simplefilter(action="ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore")


SERPER_API_KEY = os.getenv("SERPER_API_KEY", "")
USE_SEARCH_DATE_FILTER = True
SEARCH_DATE_INTERVAL_DAYS = 2


REQUEST_HEADERS = {
    "User-Agent": "Chrome/120.0",
    "Accept": "text/html",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://www.google.com/",
    "DNT": "1",
    "Upgrade-Insecure-Requests": "1",
}


def clean_bing_redirect(url):
    """Return the real page URL when Bing wraps it in a redirect link."""
    if "bing.com" not in url or "url=" not in url.lower():
        return url

    parsed_url = urllib.parse.urlparse(url)
    query_args = urllib.parse.parse_qs(parsed_url.query)
    return query_args.get("url", [url])[0]


def read_article_body(url):
    """Try to pull readable article text from a URL."""
    resolved_url = clean_bing_redirect(url)

    # First try trafilatura because it usually strips menus, ads, and sidebars cleanly.
    try:
        html = trafilatura.fetch_url(resolved_url)
        if html:
            article_text = trafilatura.extract(html)
            if article_text and len(article_text) > 200:
                return article_text[:8000]
    except Exception:
        pass

    # If extraction fails, fall back to a simple BeautifulSoup paragraph scrape.
    try:
        response = requests.get(resolved_url, headers=REQUEST_HEADERS, timeout=4, allow_redirects=True)
        if response.status_code != 200:
            return ""

        soup = BeautifulSoup(response.text, "html.parser")
        for node in soup(["script", "style", "nav", "header", "footer", "aside"]):
            node.extract()

        useful_lines = []
        for paragraph in soup.find_all(["p", "li"]):
            line = paragraph.get_text(strip=True)
            if len(line) > 30:
                useful_lines.append(line)

        return " ".join(useful_lines)[:8000]
    except Exception:
        return ""


def make_date_filter(question_text):
    """Build a Serper date filter around a YYYY-MM-DD date found in the question."""
    match = re.search(r"\b(202\d)-(\d{2})-(\d{2})\b", question_text)
    if not match:
        return ""

    year, month, day = match.groups()
    try:
        target_day = datetime.strptime(f"{year}-{month}-{day}", "%Y-%m-%d")
        span = max(0, int(SEARCH_DATE_INTERVAL_DAYS))
        start_date = (target_day - timedelta(days=span)).strftime("%m/%d/%Y")
        end_date = (target_day + timedelta(days=span)).strftime("%m/%d/%Y")
        return f"cdr:1,cd_min:{start_date},cd_max:{end_date}"
    except Exception:
        return ""


def describe_date_window(date_window):
    match = re.search(r"cd_min:([^,]+),cd_max:([^,]+)", date_window or "")
    if not match:
        return ""
    return f"{match.group(1)} to {match.group(2)}"


def gather_primary_news(search_text, date_window=""):
    print(f"\nQuery trace: Search phrase: '{search_text}'")
    if date_window:
        print(f"        Date interval: {describe_date_window(date_window)}")

    if not SERPER_API_KEY:
        print("Query trace: SERPER_API_KEY not found - skipping primary lookup.")
        return ""

    request_body = {"q": search_text, "num": 6}
    if date_window:
        request_body["tbs"] = date_window

    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}

    try:
        response = requests.post(
            "https://google.serper.dev/news",
            headers=headers,
            data=json.dumps(request_body),
            timeout=10,
        )
        if response.status_code != 200:
            return ""

        results = response.json().get("news", [])

        # If the date-filtered query is too narrow, retry once without the date window.
        if not results and date_window:
            print("        Query trace: Date interval search empty; retrying broad search.")
            request_body.pop("tbs", None)
            wide_response = requests.post(
                "https://google.serper.dev/news",
                headers=headers,
                data=json.dumps(request_body),
                timeout=10,
            )
            results = wide_response.json().get("news", [])

        # News search can miss fresh pages, so use regular web search as one more fallback.
        if not results:
            print("        Query trace: News channel empty; checking general web results.")
            web_response = requests.post(
                "https://google.serper.dev/search",
                headers=headers,
                data=json.dumps(request_body),
                timeout=10,
            )
            results = web_response.json().get("organic", [])

        top_links = [item.get("link") for item in results[:2] if item.get("link")]
        article_text_by_url = {}

        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
            futures = {executor.submit(read_article_body, link): link for link in top_links}
            for future in concurrent.futures.as_completed(futures):
                link = futures[future]
                try:
                    article_text_by_url[link] = future.result()
                except Exception:
                    article_text_by_url[link] = ""

        context_parts = []
        for item in results[:4]:
            title = item.get("title", "")
            date_label = item.get("date", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")

            block = f"Title: {title}\nDate: {date_label}\nSummary: {snippet}\n"
            if article_text_by_url.get(link):
                block += f"EXTENDED FULL TEXT: {article_text_by_url[link][:2500]}...\n"
            context_parts.append(block)

        return "\n".join(context_parts)
    except Exception as error:
        print(f"Query trace: Primary lookup error: {error}")
        return ""


class SecondaryNewsIndex:
    """Small Bing RSS + FAISS fallback used when Serper does not find enough evidence."""

    def __init__(self):
        print("Secondary index: Loading Bing RSS + vector fallback...")
        self.encoder = SentenceTransformer("all-MiniLM-L6-v2")
        self.rss_cache = {}

    def make_chunks(self, text, chunk_size=600, overlap=150):
        clean_text = re.sub(r"\s+", " ", text)
        sentences = re.split(r"(?<=[.])\s+", clean_text)

        chunks = []
        current_chunk = ""
        for sentence in sentences:
            if len(current_chunk) + len(sentence) <= chunk_size:
                current_chunk += " " + sentence
            else:
                if current_chunk.strip():
                    chunks.append(current_chunk.strip())
                current_chunk = sentence

        if current_chunk.strip():
            chunks.append(current_chunk.strip())
        return chunks

    def search_backup_index(self, search_terms, semantic_query, top_k=6):
        search_words = search_terms.split()
        search_variants = [search_terms]
        if len(search_words) > 3:
            search_variants.append(" ".join(search_words[:-1]))
        if len(search_words) > 2:
            search_variants.append(" ".join(search_words[:2]))

        candidate_chunks = []
        seen_urls = set()
        headers = {"User-Agent": "Mozilla/5.0"}

        for query in search_variants:
            if not query.strip():
                continue

            if query in self.rss_cache:
                candidate_chunks.extend(self.rss_cache[query])
                break

            chunks_for_query = []
            try:
                encoded_query = urllib.parse.quote(query)
                rss_url = f"https://www.bing.com/news/search?q={encoded_query}&format=rss"
                response = requests.get(rss_url, headers=headers, timeout=10)
                if response.status_code != 200:
                    continue

                root = ET.fromstring(response.text)
                items = root.findall(".//channel/item")

                feed_items = []
                for item in items:
                    link_node = item.find("link")
                    link = link_node.text if link_node is not None else ""
                    if link and link not in seen_urls:
                        seen_urls.add(link)
                        feed_items.append((item, link))
                    if len(feed_items) >= 3:
                        break

                def read_feed_item(feed_item):
                    item, link = feed_item
                    title_node = item.find("title")
                    desc_node = item.find("description")

                    title = title_node.text if title_node is not None else ""
                    description = desc_node.text if desc_node is not None else ""
                    description = re.sub("<[^<]+>", " ", description)

                    article_text = read_article_body(link)
                    combined_text = f"{title}. {description}. {article_text}"
                    return self.make_chunks(combined_text)

                with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
                    for chunks in executor.map(read_feed_item, feed_items):
                        chunks_for_query.extend(chunks or [])

                self.rss_cache[query] = chunks_for_query
                candidate_chunks.extend(chunks_for_query)
                if candidate_chunks:
                    break
            except Exception:
                continue

        if not candidate_chunks:
            return ""

        # Remove near-duplicate chunks before embedding them.
        deduped_chunks = []
        chunk_signatures = set()
        for chunk in candidate_chunks:
            signature = chunk[:100].strip()
            if signature not in chunk_signatures:
                chunk_signatures.add(signature)
                deduped_chunks.append(chunk)

        if not deduped_chunks:
            return ""

        embeddings = self.encoder.encode(deduped_chunks, convert_to_numpy=True)
        index = faiss.IndexFlatL2(embeddings.shape[1])
        index.add(embeddings)

        question_vector = self.encoder.encode([semantic_query], convert_to_numpy=True)
        _, nearest_ids = index.search(question_vector, min(top_k, len(deduped_chunks)))

        selected_chunks = [deduped_chunks[index_id] for index_id in nearest_ids[0][:6]]
        return "\n\n".join(selected_chunks)


secondary_news_index = SecondaryNewsIndex()


def wrap_output_text(text, indent="", subsequent_indent=None):
    subsequent_indent = indent if subsequent_indent is None else subsequent_indent
    return textwrap.fill(
        str(text or ""),
        width=100,
        initial_indent=indent,
        subsequent_indent=subsequent_indent,
        break_long_words=False,
        break_on_hyphens=False,
    )


def display_question(question, question_count, level, seconds_left=None):
    border = "=" * 80
    time_label = "" if seconds_left is None else f" | {seconds_left:.1f}s left"

    print("\n" + border)
    print(f"Question {question_count} | Level {level}{time_label}")
    print(wrap_output_text(question.text))

    for index, option in enumerate(question.options):
        answer_letter = chr(65 + index)
        line = f"{answer_letter}: {option.id}: {option.text}"
        print(wrap_output_text(line, indent="  ", subsequent_indent="     "))

    print(border)


def pull_eval_summary(text):
    match = re.search(r"Eval:\s*(.+?)(?:\n|FINAL ANSWER:|$)", str(text or ""), re.IGNORECASE | re.DOTALL)
    if not match:
        return ""
    return re.sub(r"\s+", " ", match.group(1)).strip()


def make_search_keywords(question_text, options):
    prompt = f"""[INST] You are a specialist in news-search query construction.
Create a focused Google News query of no more than 5 words that can locate the exact article.

Rules:
1. Use only the core event, distinctive proper nouns, and main subjects from the question.
2. Do not include or borrow wording from the answer choices.
3. Return bare noun keywords separated by spaces. No verbs, punctuation, or explanation.

Generate keywords for this question:
{question_text}
Keywords: [/INST]"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=25,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    keywords = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    keywords = re.sub(r"\[/?INST\]|Keywords?:", " ", keywords, flags=re.IGNORECASE)
    keywords = re.sub(r"[^\w\s\-']", " ", keywords)
    return re.sub(r"\s+", " ", keywords).strip()


def read_final_letter(text):
    upper_text = text.upper()
    if "FINAL ANSWER: NONE" in upper_text:
        return "N"

    match = re.search(r"FINAL ANSWER:\s*([ABCD])", text, re.IGNORECASE)
    if match:
        return match.group(1).upper()

    letters = re.findall(r"\b([ABCD])\b", upper_text)
    return letters[-1] if letters else "A"


def ask_news_model(context, question):
    prompt = f"""[INST] You are a careful current-events analyst answering a multiple-choice question from retrieved news context.

Context:
{context}

Question:
{question.text}
A) {question.options[0].text}
B) {question.options[1].text}
C) {question.options[2].text}
D) {question.options[3].text}

Instructions:
1. Choose the option best supported by the evidence.
2. If the question asks for what is NOT, EXCEPT, false, missing, or denied, choose the option that the text excludes or fails to support.
3. If several options appear as nested locations or entities, choose the most specific one.
4. Combine details across snippets when that is necessary to identify the correct option.
5. If the context does not support an answer, output 'FINAL ANSWER: NONE'. Do not guess.
6. After 'FINAL ANSWER:' output only A, B, C, D, or NONE. Do not write the option text.

Use exactly this format:
Eval: justify the selected option in at most 15 words
FINAL ANSWER: A, B, C, D, or NONE
[/INST]Eval: """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    reply = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    reply = re.sub(r"\[/?INST\]", " ", reply, flags=re.IGNORECASE)
    reply = re.sub(r"\s+", " ", reply).strip()
    return re.sub(r"\s*FINAL ANSWER:", "\nFINAL ANSWER:", reply, flags=re.IGNORECASE).strip()


def select_answer(question):
    if len(question.options) < 4:
        return question.options[0].id, "A", "fallback"

    search_terms = make_search_keywords(question.text, question.options)
    date_filter = make_date_filter(question.text) if USE_SEARCH_DATE_FILTER else ""

    answer_letter = "N"
    model_reply = ""

    # Primary path: look for direct news evidence first.
    context = gather_primary_news(search_terms, date_filter)
    if context.strip():
        print("\n" + "-" * 40)
        print("Source packet A: Primary Serper evidence:")
        print(context)
        print("-" * 40 + "\n")

        model_reply = ask_news_model(context, question)
        answer_letter = read_final_letter(model_reply)
        print(f"Model read A:\n{model_reply}")

    # Secondary path: if the model says NONE, gather RSS evidence and rank it semantically.
    if answer_letter == "N":
        print("\nSecondary pass: Primary answer was NONE; checking fallback evidence...")
        option_text = " ".join(option.text for option in question.options)
        semantic_query = f"{question.text} {option_text}"
        backup_evidence = secondary_news_index.search_backup_index(
            search_terms=search_terms,
            semantic_query=semantic_query,
            top_k=6,
        )

        if backup_evidence.strip():
            print("\n" + "-" * 40)
            print("Source packet B: Bing RSS fallback evidence:")
            print(backup_evidence)
            print("-" * 40 + "\n")

            model_reply = ask_news_model(backup_evidence, question)
            answer_letter = read_final_letter(model_reply)
            print(f"Model read B:\n{model_reply}")
        else:
            print("\nSecondary pass: No fallback evidence found.")

    # Keep the caller from crashing when neither retrieval path supports an answer.
    if answer_letter == "N":
        print("\nFinal fallback: No supported answer found; using option A.")
        answer_letter = "A"

    try:
        choice_index = ["A", "B", "C", "D"].index(answer_letter)
    except ValueError:
        choice_index = 0
        answer_letter = "A"

    return question.options[choice_index].id, answer_letter, model_reply


In [ ]:
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError


def play_game(competition_id=5, mode='text'):
    API_URL = (__import__("os").getenv("POLI_MILLIONAIRE_API_URL") or input("PoliMillionaire API URL: ").strip())
    client = MillionaireClient(API_URL)
    user = client.login((__import__("os").getenv("POLI_MILLIONAIRE_USERNAME") or input("PoliMillionaire username: ").strip()), (__import__("os").getenv("POLI_MILLIONAIRE_PASSWORD") or __import__("getpass").getpass("PoliMillionaire password: ").strip()))
    print(f"Session user: {user.username}")



    game = client.game.start(competition_id=competition_id, mode=mode)
    question_count = 0
    correct_answers = 0

    while game.in_progress:
        question = game.current_question
        if question is None:
            break

        question_count += 1
        seconds_left = getattr(game, "time_remaining", None)
        display_question(question, question_count, game.current_level, seconds_left)

        t0 = time.time()
        option_id, answer_letter, answer_trace = select_answer(question)
        t1 = time.time()

        chosen_offset = builtins.max(0, ord(answer_letter) - ord("A")) if answer_letter in "ABCD" else 0
        chosen_text = question.options[chosen_offset].text if chosen_offset < len(question.options) else ""
        print("-" * 80)
        print(wrap_output_text(f"Selected answer: {answer_letter}: {option_id}: {chosen_text}"))
        brief_note = pull_eval_summary(answer_trace)
        if brief_note:
            print(wrap_output_text(f"Brief rationale: {brief_note}"))
        print(f"Processing time: {t1-t0:.2f}s")

        try:
            result = game.answer(option_id)
            if result.correct:
                correct_answers += 1
            print(f"Outcome: {'CORRECT' if result.correct else 'WRONG'} | Correct answers: {correct_answers}")
        except TimeoutError:
            print("Outcome: TIMED OUT | generation took more than thirty seconds")
            break
        except RateLimitError:
            print("Rate limited; waiting five seconds before retrying submit.")
            time.sleep(5)
            result = game.answer(option_id)
            if result.correct:
                correct_answers += 1
            print(f"Outcome: {'CORRECT' if result.correct else 'WRONG'} | Correct answers: {correct_answers}")

        if result.game_over:
            break
        time.sleep(1)

    print(f"Final correct answers: {correct_answers}")
    game.correct_answers = correct_answers
    return game

In [ ]:
game = play_game(competition_id=5, mode='text')